# Baselines: established ML and DL time series classifiers

Reproduces the comparison in Section 5.2 of the paper. Five classifiers from
`sktime` are evaluated with 5-fold cross-validation over all 777 labeled lakes;
each fold trains on about 622 lakes and predicts the held-out fifth, and
accuracy is computed over the pooled out-of-fold predictions.

Unlike RPS-GMM, which trains on three lakes in total, every model here sees 80%
of the dataset in each fold.

This notebook imports its implementation from
[`scripts/run_baselines.py`](../../scripts/run_baselines.py) so there is a
single definition of the protocol. To run everything non-interactively instead:

```bash
python scripts/run_baselines.py --features backscatter
python scripts/run_baselines.py --features backscatter_water
```

> **Environment.** The baselines need the optional `baselines` extra
> (`pip install -e ".[baselines]"`), which pulls in `sktime`, `tensorflow` and
> `dtaidistance`. Install it in its own environment: `sktime` pins an older
> `scikit-learn` than the core pipeline uses.

> **Runtime.** The four deep models train for 100 epochs across 5 folds. Budget
> a few hours on CPU for both feature sets; a GPU shortens this considerably.
> The nearest-neighbour baseline takes a couple of minutes.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src" / "rpsgmm").is_dir():
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError("Run this notebook from inside the repository.")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "scripts"))

print("Repository root:", REPO_ROOT)

In [ ]:
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import KFold

from rpsgmm import CLASSES
from run_baselines import (
    BATCH_SIZE,
    MODELS,
    N_EPOCHS,
    N_SPLITS,
    RANDOM_STATE,
    DtwOneNearestNeighbor,
    build_model,
    load_panel,
)

warnings.filterwarnings("ignore")
print(f"{N_SPLITS}-fold CV, {N_EPOCHS} epochs, batch size {BATCH_SIZE}")

## Load

`load_panel` returns the data twice: once in sktime's nested panel format for
the deep models, and once as a plain 2-D array for the nearest-neighbour
baseline. Labels are integer codes indexing `CLASSES`.

The combined setting concatenates both channels into a single 488-step
univariate series, reproducing the published configuration.

In [ ]:
FEATURES = "backscatter"   # or "backscatter_water"

X, X_flat, y = load_panel(FEATURES)

print(f"{len(X)} lakes, {X_flat.shape[1]} time steps")
print(pd.Series([CLASSES[c] for c in y]).value_counts().to_string())

## Evaluate

In [ ]:
cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
labels = list(range(len(CLASSES)))


def report(name, predictions):
    accuracy = accuracy_score(y, predictions)
    print(f"=== {name}: {accuracy * 100:.2f}% ===")
    print(classification_report(
        y, predictions, labels=labels, target_names=list(CLASSES),
        digits=4, zero_division=0,
    ))
    return {"model": name, "accuracy": accuracy}

### Nearest neighbour

1-NN under an unconstrained DTW distance. `DtwOneNearestNeighbor` uses
`dtaidistance`'s C backend, which is orders of magnitude faster than evaluating
the same distance through sktime and gives identical nearest neighbours.

In [ ]:
predictions = np.empty(len(y), dtype=int)
for fold, (train_idx, test_idx) in enumerate(cv.split(X_flat), start=1):
    model = DtwOneNearestNeighbor().fit(X_flat[train_idx], y[train_idx])
    predictions[test_idx] = model.predict(X_flat[test_idx])
    print(f"fold {fold}/{N_SPLITS} done")

results = [report(MODELS["knn"], predictions)]

### Deep models

Each is trained from scratch on every fold, so this cell is the slow one.

In [ ]:
for name in ("lstmfcn", "fcn", "resnet", "rnn"):
    predictions = build_model(name, n_epochs=N_EPOCHS).fit_predict(X=X, y=y, cv=cv)
    results.append(report(MODELS[name], np.asarray(predictions, dtype=int)))

In [ ]:
pd.DataFrame(results).sort_values("accuracy", ascending=False, ignore_index=True)